# Python Interview Prep — Colab Cheat Sheet

A runnable notebook for practical Python data interviews: file parsing, aggregation, joins, metrics, formatting, edge cases, and mock-problem structure.

**Interview loop to say out loud:**

> “I’ll first inspect the data shape, then write the smallest transformation, then sanity-check the output against a tiny example.”

Use this notebook as a pattern library, not a memorised script.

## 0. Setup — imports, sample files, and tiny datasets

This notebook is self-contained for Colab. The first cell creates small JSON/CSV/JSONL fixtures so every technique can run immediately.

In [ ]:
from pathlib import Path
import csv
import json
import math
import statistics as stats
from collections import Counter, defaultdict
from pprint import pprint

try:
    import pandas as pd
except ImportError:
    pd = None

DATA_DIR = Path("/content/pyprep_data") if Path("/content").exists() else Path("pyprep_data")
DATA_DIR.mkdir(exist_ok=True)

sample_results = [
    {"provider": "openai", "model": "gpt-4.1", "latency_ms": 820, "cost_usd": 0.042, "score": 0.91, "status": "ok"},
    {"provider": "anthropic", "model": "sonnet", "latency_ms": 960, "cost_usd": 0.038, "score": 0.89, "status": "ok"},
    {"provider": "openai", "model": "gpt-4.1-mini", "latency_ms": 410, "cost_usd": 0.006, "score": 0.78, "status": "ok"},
    {"provider": "local", "model": "qwen", "latency_ms": 220, "cost_usd": 0.000, "score": None, "status": "timeout"},
]
(DATA_DIR / "sample_results.json").write_text(json.dumps(sample_results, indent=2))

endpoint_logs = """provider,model,latency_ms,cost_usd,score,status
openai,gpt-4.1,820,0.042,0.91,ok
anthropic,sonnet,960,0.038,0.89,ok
openai,gpt-4.1-mini,410,0.006,0.78,ok
local,qwen,220,0.000,,timeout
openai,gpt-4.1-mini,450,0.006,0.82,ok
anthropic,haiku,390,0.004,0.73,ok
"""
(DATA_DIR / "endpoint_logs.csv").write_text(endpoint_logs)

jsonl_lines = [
    {"request_id":"r1", "provider":"openai", "tokens":1200, "ok":True},
    {"request_id":"r2", "provider":"anthropic", "tokens":980, "ok":True},
    "{bad json line",
    {"request_id":"r3", "provider":"openai", "tokens":1500, "ok":False},
]
with open(DATA_DIR / "batch_outputs.jsonl", "w") as f:
    for row in jsonl_lines:
        f.write(row if isinstance(row, str) else json.dumps(row))
        f.write("\n")

metadata = {
    "gpt-4.1": {"provider": "openai", "tier": "premium"},
    "gpt-4.1-mini": {"provider": "openai", "tier": "cheap"},
    "sonnet": {"provider": "anthropic", "tier": "premium"},
    "haiku": {"provider": "anthropic", "tier": "cheap"},
    "qwen": {"provider": "local", "tier": "free"},
}
(DATA_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2))

print("Fixtures written to", DATA_DIR)

## 1. Shape inspection — always do this before solving

Interviewers reward visible reasoning. Before transforming anything, inspect type, length, columns/keys, nulls, and a sample.

In [ ]:
def inspect_records(records, name="records", n=3):
    print(f"--- {name} ---")
    print("type:", type(records).__name__)
    try:
        print("len:", len(records))
    except TypeError:
        print("len: <not available>")
    if isinstance(records, list) and records:
        print("first type:", type(records[0]).__name__)
        if isinstance(records[0], dict):
            print("keys:", sorted(records[0].keys()))
        print("sample:")
        pprint(records[:n])
    elif isinstance(records, dict):
        print("keys:", sorted(records.keys()))
        pprint(dict(list(records.items())[:n]))

records = json.load(open(DATA_DIR / "sample_results.json"))
inspect_records(records, "sample_results")

## 2. JSON pattern — load and summarize safely

Use `json.load(file_object)` for files and `json.loads(string)` for strings. Avoid touching `data[0]` until you know the list is non-empty.

In [ ]:
def load_json_summary(path: str) -> dict:
    with open(path) as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("Expected a JSON list of records")

    first = data[0] if data else {}
    fields = sorted(first.keys()) if isinstance(first, dict) else []
    return {
        "total_records": len(data),
        "fields": fields,
        "sample": data[:2],
    }

summary = load_json_summary(DATA_DIR / "sample_results.json")
pprint(summary)

## 3. CSV pattern — stdlib and pandas

`csv.DictReader` is excellent in interviews because it makes columns explicit. Pandas is faster for analytics once the shape is clear.

In [ ]:
def read_csv_dicts(path: str) -> list[dict]:
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

rows = read_csv_dicts(DATA_DIR / "endpoint_logs.csv")
inspect_records(rows, "csv rows")

# Convert numeric fields deliberately; CSV values start as strings.
for row in rows:
    row["latency_ms"] = int(row["latency_ms"])
    row["cost_usd"] = float(row["cost_usd"])
    row["score"] = float(row["score"]) if row["score"] else None

pprint(rows[:2])

In [ ]:
if pd is not None:
    df = pd.read_csv(DATA_DIR / "endpoint_logs.csv")
    print(df.head())
    print("\ndtypes:")
    print(df.dtypes)
    print("\nnulls:")
    print(df.isna().sum())
else:
    print("pandas not available")

## 4. JSONL pattern — parse line by line and report bad lines

For JSONL, one malformed line should not necessarily kill the whole file. Return parsed records plus a bad-line report.

In [ ]:
def parse_jsonl_with_report(path: str) -> dict:
    records = []
    bad_lines = []
    with open(path) as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                bad_lines.append({"line_no": line_no, "error": str(e), "preview": line[:80]})
    return {"records": records, "bad_line_count": len(bad_lines), "bad_lines": bad_lines}

jsonl_report = parse_jsonl_with_report(DATA_DIR / "batch_outputs.jsonl")
pprint(jsonl_report)

## 5. Aggregation — groupby with stdlib

If pandas is unavailable or overkill, use `defaultdict`, `Counter`, and small helper functions.

In [ ]:
def mean(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

def groupby_provider_stdlib(rows: list[dict]) -> list[dict]:
    groups = defaultdict(list)
    for row in rows:
        groups[row["provider"]].append(row)

    output = []
    for provider, items in groups.items():
        ok_items = [r for r in items if r.get("status") == "ok"]
        output.append({
            "provider": provider,
            "requests": len(items),
            "ok_rate": round(len(ok_items) / len(items), 3),
            "avg_latency_ms": round(mean([r["latency_ms"] for r in items]), 1),
            "avg_score": None if mean([r["score"] for r in items]) is None else round(mean([r["score"] for r in items]), 3),
            "total_cost_usd": round(sum(r["cost_usd"] for r in items), 4),
        })
    return sorted(output, key=lambda r: r["provider"])

pprint(groupby_provider_stdlib(rows))

## 6. Aggregation — pandas groupby

Use pandas when you need fast tabular transformations, sorting, joins, percentiles, and display formatting.

In [ ]:
if pd is not None:
    df = pd.read_csv(DATA_DIR / "endpoint_logs.csv")
    summary_df = (
        df.assign(is_ok=df["status"].eq("ok"))
          .groupby("provider", as_index=False)
          .agg(
              requests=("model", "size"),
              ok_rate=("is_ok", "mean"),
              avg_latency_ms=("latency_ms", "mean"),
              avg_score=("score", "mean"),
              total_cost_usd=("cost_usd", "sum"),
          )
          .sort_values(["avg_score", "ok_rate"], ascending=False)
    )
    display(summary_df)

## 7. Metrics — mean, median, percentiles, pass rates

Be explicit about null policy. In interviews, say whether you skip nulls, coerce them, or treat them as failures.

In [ ]:
def percentile(values, p):
    """Nearest-rank-ish percentile with interpolation via statistics.quantiles alternative."""
    values = sorted(v for v in values if v is not None)
    if not values:
        return None
    k = (len(values) - 1) * (p / 100)
    lo = math.floor(k)
    hi = math.ceil(k)
    if lo == hi:
        return values[int(k)]
    return values[lo] + (values[hi] - values[lo]) * (k - lo)

latencies = [r["latency_ms"] for r in rows]
scores = [r["score"] for r in rows if r["score"] is not None]
metrics = {
    "latency_mean": round(stats.mean(latencies), 1),
    "latency_median": stats.median(latencies),
    "latency_p95": round(percentile(latencies, 95), 1),
    "score_mean": round(stats.mean(scores), 3),
    "pass_rate_score_080": round(sum(s >= 0.80 for s in scores) / len(scores), 3),
}
pprint(metrics)

## 8. Joins and metadata enrichment

Always check cardinality. Say what you expect: many log rows to one metadata row by `model`.

In [ ]:
metadata = json.load(open(DATA_DIR / "model_metadata.json"))

def enrich_rows_with_metadata(rows, metadata):
    enriched = []
    missing_models = set()
    for row in rows:
        meta = metadata.get(row["model"])
        if meta is None:
            missing_models.add(row["model"])
            meta = {"tier": "unknown"}
        enriched.append({**row, **{f"model_{k}": v for k, v in meta.items()}})
    return enriched, sorted(missing_models)

enriched, missing = enrich_rows_with_metadata(rows, metadata)
print("missing models:", missing)
pprint(enriched[:3])

In [ ]:
if pd is not None:
    df = pd.read_csv(DATA_DIR / "endpoint_logs.csv")
    meta_df = pd.DataFrame([
        {"model": model, **attrs} for model, attrs in metadata.items()
    ])
    merged = df.merge(meta_df, on="model", how="left", validate="many_to_one")
    display(merged)

## 9. Output formatting — make results human-readable

Interviewers like deliberate presentation: rename columns, round values, order columns, and sort by the decision metric.

In [ ]:
def format_provider_report(summary_rows):
    report = []
    for r in summary_rows:
        report.append({
            "Provider": r["provider"],
            "Requests": r["requests"],
            "OK Rate": f"{r['ok_rate']:.0%}",
            "Avg Latency (ms)": r["avg_latency_ms"],
            "Avg Score": "n/a" if r["avg_score"] is None else f"{r['avg_score']:.3f}",
            "Total Cost ($)": f"{r['total_cost_usd']:.4f}",
        })
    return report

report = format_provider_report(groupby_provider_stdlib(rows))
pprint(report)

## 10. Edge cases — nulls, deduplication, anomalies

Name your policy before coding it. Example: “Null scores are excluded from score averages but count against availability if status is not ok.”

In [ ]:
def null_report(rows):
    keys = sorted({k for row in rows for k in row.keys()})
    return {k: sum(row.get(k) in (None, "", "null") for row in rows) for k in keys}

pprint(null_report(rows))

# Deduplicate by stable key. Here we simulate duplicate request IDs.
dup_rows = [
    {"request_id":"r1", "score":0.9, "updated_at":1},
    {"request_id":"r1", "score":0.92, "updated_at":2},
    {"request_id":"r2", "score":0.7, "updated_at":1},
]

def dedupe_latest(rows, key="request_id", version="updated_at"):
    best = {}
    for row in rows:
        if row[key] not in best or row[version] > best[row[key]][version]:
            best[row[key]] = row
    return list(best.values())

pprint(dedupe_latest(dup_rows))

# Simple anomaly flag: latency greater than 2x median.
median_latency = stats.median(latencies)
anomalies = [r for r in rows if r["latency_ms"] > 2 * median_latency]
print("median latency:", median_latency)
pprint(anomalies)

## 11. Debug workflow — when the code fails

Use this sequence instead of panicking:

1. Re-read the error message.
2. Print the type and first sample of the object that failed.
3. Isolate one tiny input.
4. Add one assertion for the assumption.
5. Re-run the smallest failing function.

In [ ]:
def assert_record_shape(row):
    required = {"provider", "model", "latency_ms", "cost_usd", "score", "status"}
    missing = required - set(row)
    assert not missing, f"Missing keys: {missing}. Row preview: {row}"

def validate_rows(rows):
    for i, row in enumerate(rows):
        try:
            assert_record_shape(row)
            assert isinstance(row["latency_ms"], int), f"latency_ms must be int, got {type(row['latency_ms']).__name__}"
        except AssertionError as e:
            raise AssertionError(f"Row {i} failed validation: {e}")

validate_rows(rows)
print("rows validated")

## 12. Mock problem template — 20/30/45-minute structure

Copy this structure in a real interview. It keeps your work visible and testable.

In [ ]:
def solve_provider_report(csv_path: str, metadata_path: str) -> list[dict]:
    """
    Interview-ready structure:
    1. Load files
    2. Inspect/validate shape
    3. Normalize types
    4. Enrich/join metadata
    5. Aggregate metrics
    6. Format output
    7. Sanity-check invariants
    """
    rows = read_csv_dicts(csv_path)
    for row in rows:
        row["latency_ms"] = int(row["latency_ms"])
        row["cost_usd"] = float(row["cost_usd"])
        row["score"] = float(row["score"]) if row["score"] else None

    metadata = json.load(open(metadata_path))
    enriched, missing = enrich_rows_with_metadata(rows, metadata)
    if missing:
        print("Warning: missing metadata for", missing)

    grouped = groupby_provider_stdlib(enriched)
    output = format_provider_report(grouped)

    assert sum(r["requests"] for r in grouped) == len(rows)
    return output

final_report = solve_provider_report(DATA_DIR / "endpoint_logs.csv", DATA_DIR / "model_metadata.json")
pprint(final_report)

## 13. Narration phrases to keep beside you

- “I’m going to inspect the shape before assuming the schema.”
- “I’ll make the null policy explicit before calculating the metric.”
- “I expect this join to be many log rows to one metadata row.”
- “I’ll sort by the metric that answers the business question.”
- “Let me sanity-check the row count and one hand-calculated example.”
- “If this were production, I’d add validation around this boundary.”

## 14. Final interview checklist

- [ ] Did I inspect input shape?
- [ ] Did I convert types deliberately?
- [ ] Did I state null/bad-line policy?
- [ ] Did I validate join cardinality?
- [ ] Did I calculate at least one metric by hand or with a tiny sample?
- [ ] Did I format output for a human reviewer?
- [ ] Did I narrate tradeoffs and assumptions?